# Coin Flip Trading System

Uses a coin flip based trading strategy (heads = long, tails = short) to make trading decisions at market open while leveraging timeseries forcasting

## Module.1
### dependencies

In [21]:
# cellblock.1

# libraries
import numpy as np
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels
import yfinance as yf
import math
import re

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "font.size": 9})

In [ ]:
# cellblock.2

# import local data
#
# Only the PATH is needed here. The file is ~0.74 GB / 6.9M rows, and reading
# all of it just to print five lines costs a couple of GB that nothing
# downstream ever touches - Module.2 cellblock.1 re-reads it properly with
# usecols and narrow dtypes. So this cellblock peeks and stops.
import os

file_path = "/Users/andrew/main/trading/data/NQ.1m.OHLCV.data/NQ-2010.06.06-2026.01.23-ohlcv-1m.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"local /NQ data not found at {file_path}")

df = pd.read_csv(file_path, nrows=5)          # PREVIEW ONLY - not the dataset
print(f"{os.path.basename(file_path)}")
print(f"  size    : {os.path.getsize(file_path) / 1e9:.2f} GB on disk")
print(f"  columns : {list(df.columns)}")
print(df.head())


In [ ]:
# cellblock.3

# load yfinance data
YF_PERIOD   = "720d"        # just inside Yahoo's 730-day cap at 1h (~2 yrs)
YF_INTERVAL = "1h"
OHLCV = ["open", "high", "low", "close", "volume"]


def fetch_1h(ticker, label, period=YF_PERIOD):
    """Download max-available 1h bars and return a clean OHLCV frame.

    Returns all five OHLCV columns; index is tz-aware and sorted ascending.
    Yahoo serves at most 730 days at 1h against 60 days at 15m, so moving to the
    hourly interval buys ~12x the history - that depth is the reason to use it.
    720d is requested rather than 730d: it sits just inside the cap, so a
    boundary-rounding day at the far end cannot turn the whole request into an
    error. Yahoo often returns MORE than the window asked for anyway.
    """
    # prepost=True is what makes an equity tradeable by this system at all.
    # Yahoo's REGULAR-hours 1h bars for a stock start at 09:30, so nothing exists
    # in the 08:30-09:30 window the entry rule needs and every session is skipped.
    # With pre/post on, the 04:00-09:30 bars appear and the 09:00 bar lands in
    # the window. Verified to be a NO-OP for futures, crypto and ^VIX (identical
    # bar counts on NQ=F, ES=F, RTY=F, YM=F, BTC-USD, ETH-USD, ^VIX) - it only
    # changes the three equities, so it is safe to leave on for everything.
    # Caveat that matters: Yahoo reports pre/post VOLUME as 0. The prices are
    # real, the liquidity is not measurable here, and pre-market spreads are far
    # wider than the 1-tick slippage cellblock.2 assumes.
    raw = yf.download(ticker, period=period, interval=YF_INTERVAL,
                      auto_adjust=False, progress=False, threads=False,
                      prepost=True)
    if raw is None or len(raw) == 0:
        raise RuntimeError(f"{label} ({ticker}): yfinance returned no rows")

    # yfinance returns MultiIndex columns (field, ticker) -> flatten to field
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(-1)

    df = raw.rename(columns=str.lower)
    missing = [c for c in OHLCV if c not in df.columns]
    if missing:
        raise RuntimeError(f"{label} ({ticker}): missing columns {missing}")

    df = df[OHLCV].copy()
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df = df.dropna(subset=["open", "high", "low", "close"])
    df["volume"] = df["volume"].fillna(0)
    df.index.name = "ts"
    df.columns.name = None
    return df


def describe_1h(df, label, ticker):
    """Print coverage + integrity checks. Returns the frame unchanged."""
    span_days = (df.index[-1] - df.index[0]).total_seconds() / 86400
    bad_hl = int((df["high"] < df["low"]).sum())
    bad_rng = int((~((df["high"] >= df[["open", "close"]].max(axis=1) - 1e-6) &
                     (df["low"]  <= df[["open", "close"]].min(axis=1) + 1e-6))).sum())
    gap = df.index.to_series().diff().dt.total_seconds().div(60)
    print(f"{label}  ({ticker})  interval=1h")
    print(f"  bars      : {len(df):,}")
    print(f"  coverage  : {df.index[0]}  ->  {df.index[-1]}   ({span_days:.1f} calendar days)")
    print(f"  columns   : {list(df.columns)}")
    print(f"  close     : {df['close'].min():,.2f}  -  {df['close'].max():,.2f}")
    print(f"  volume    : total {df['volume'].sum():,.0f}"
          + ("   [!] all-zero: Yahoo reports no volume for this symbol"
             if df["volume"].sum() == 0 else ""))
    print(f"  integrity : high<low {bad_hl} | OHLC out of range {bad_rng} | NaN {int(df.isna().sum().sum())}")
    print(f"  bar gaps  : median {gap.median():.0f}m, max {gap.max():,.0f}m "
          f"(>60m gaps are session breaks/weekends)")
    return df


def plot_ohlcv(df, label, ticker, interval="1h", bar_minutes=60, downsampled=False):
    """Two-panel OHLCV chart: high-low range + close on top, volume beneath."""
    fig, (ax1, ax2) = plt.subplots(
        2, 1, sharex=True, figsize=(13, 6.5),
        gridspec_kw={"height_ratios": [3, 1]})

    ax1.fill_between(df.index, df["low"], df["high"], color="#4a7ebb",
                     alpha=0.30, linewidth=0, label="High-Low range")
    ax1.plot(df.index, df["close"], color="#14375e", linewidth=0.8, label="Close")
    ax1.plot(df.index, df["open"],  color="#c0504d", linewidth=0.5,
             alpha=0.55, label="Open")
    ax1.set_ylabel("Price")
    ax1.set_title(f"{label}  ({ticker})  -  {interval} OHLCV"
                  + ("  [chart downsampled]" if downsampled else "")
                  + f"  |  {len(df):,} bars  |  "
                  f"{df.index[0]:%Y-%m-%d} to {df.index[-1]:%Y-%m-%d}")
    ax1.legend(loc="upper left", fontsize=8, framealpha=0.9)

    width = bar_minutes / (24 * 60) * 0.9
    ax2.bar(df.index, df["volume"], width=width, color="#7f7f7f", linewidth=0)
    ax2.set_ylabel("Volume")
    ax2.set_xlabel("Time")
    if df["volume"].sum() == 0:
        ax2.text(0.5, 0.5, "no volume reported by source", ha="center",
                 va="center", transform=ax2.transAxes, fontsize=9, color="#c0504d")
    fig.autofmt_xdate()
    fig.subplots_adjust(hspace=0.08)
    plt.show()


def load_plot_1h(ticker, label):
    """Fetch -> validate -> chart. Returns the full 1h OHLCV frame."""
    df = fetch_1h(ticker, label)
    describe_1h(df, label, ticker)
    plot_ohlcv(df, label, ticker)
    return df


## Module.2
### load assets
**BTC, /ES, /NQ, /RTY, /YM, ETH, AAPL, NVDA, TSLA, VIX**

| cellblock | asset | source | interval | variable |
|---|---|---|---|---|
| 1 | /NQ | local CSV (front-month continuous) | **1m** | `nq_1m` |
| 2 | /NQ | yfinance `NQ=F` | 1h | `nq_1h` |
| 3 | /ES | yfinance `ES=F` | 1h | `es_1h` |
| 4 | /RTY | yfinance `RTY=F` | 1h | `rty_1h` |
| 5 | /YM | yfinance `YM=F` | 1h | `ym_1h` |
| 6 | BTC | yfinance `BTC-USD` | 1h | `btc_1h` |
| 7 | ETH | yfinance `ETH-USD` | 1h | `eth_1h` |
| 8 | TSLA | yfinance `TSLA` | 1h | `tsla_1h` |
| 9 | NVDA | yfinance `NVDA` | 1h | `nvda_1h` |
| 10 | AAPL | yfinance `AAPL` | 1h | `aapl_1h` |
| 11 | VIX | yfinance `^VIX` | 1h | `vix_1h` |

Every frame carries the full `open, high, low, close, volume` set. Yahoo caps 1h
intraday history at 730 days; `period="720d"` sits just inside that cap - about
12x the 60-day window it allows at 15m. Realised coverage varies by asset (crypto 24/7 > futures ~23h > equities RTH only).

In [ ]:
# cellblock.1

# /NQ local data (1m)
#
# Prerequisites, checked up front so a missing one says WHICH cell to run
# instead of surfacing as a bare NameError 40 lines in:
#   Module.1 cellblock.1 -> np, pd, plt, re
#   Module.1 cellblock.2 -> file_path
#   Module.1 cellblock.3 -> plot_ohlcv, OHLCV
_need = {"file_path": "Module.1 cellblock.2", "plot_ohlcv": "Module.1 cellblock.3",
         "OHLCV": "Module.1 cellblock.3", "re": "Module.1 cellblock.1",
         "np": "Module.1 cellblock.1", "pd": "Module.1 cellblock.1"}
_missing = {n: c for n, c in _need.items() if n not in globals()}
if _missing:
    raise RuntimeError("run these first -> " + "; ".join(
        f"{c} (defines {n})" for n, c in _missing.items()))

OUTRIGHT = re.compile(r"^NQ[HMUZ]\d$")          # NQ + month code + year digit
NQ_COLS  = ["ts_event", "instrument_id", "open", "high", "low", "close", "volume", "symbol"]

nq_raw = pd.read_csv(
    file_path, usecols=NQ_COLS, engine="pyarrow",
    dtype={"instrument_id": "int64", "open": "float32", "high": "float32",
           "low": "float32", "close": "float32", "volume": "int32",
           "symbol": "category"})

n_all = len(nq_raw)
nq_raw = nq_raw[nq_raw["symbol"].astype(str).str.fullmatch(OUTRIGHT)].copy()
print(f"rows: {n_all:,} total -> {len(nq_raw):,} outright "
      f"({n_all - len(nq_raw):,} spread rows dropped, {100*(n_all-len(nq_raw))/n_all:.2f}%)")

nq_raw["ts_event"] = pd.to_datetime(nq_raw["ts_event"], utc=True, format="ISO8601")
nq_raw["date"] = nq_raw["ts_event"].dt.date

# expiry order: each instrument_id is one real contract, so its last bar orders it
expiry_rank = {cid: k for k, cid in enumerate(
    nq_raw.groupby("instrument_id", observed=True)["ts_event"].max().sort_values().index)}
rank_to_id = {k: cid for cid, k in expiry_rank.items()}
print(f"distinct contracts (by instrument_id): {len(expiry_rank)}")

# front month = highest daily volume, constrained to never roll backwards
daily_vol = (nq_raw.groupby(["date", "instrument_id"], observed=True)["volume"]
                   .sum().reset_index())
front = (daily_vol.loc[daily_vol.groupby("date", observed=True)["volume"].idxmax(),
                       ["date", "instrument_id"]]
                  .rename(columns={"instrument_id": "front"})
                  .sort_values("date").reset_index(drop=True))
front["rank"] = front["front"].map(expiry_rank).astype(int)
front["rank_fwd"] = front["rank"].cummax()
print(f"days where raw volume leader moved backwards: {(front['rank'] != front['rank_fwd']).sum()}")
front["front_id"] = front["rank_fwd"].map(rank_to_id)
roll_dates = front.loc[front["front_id"] != front["front_id"].shift(), "date"].tolist()
print(f"roll events: {len(roll_dates)}  (expect ~4/yr over "
      f"{(nq_raw['ts_event'].max() - nq_raw['ts_event'].min()).days/365.25:.1f} yrs)")

# keep only bars belonging to that day's front contract -> continuous 1m OHLCV
nq_1m = (nq_raw.merge(front[["date", "front_id"]], on="date", how="left")
               .query("instrument_id == front_id")
               .sort_values("ts_event")
               .set_index("ts_event")[["open", "high", "low", "close", "volume", "symbol"]])
nq_1m.index.name = "ts"

# ---- integrity checks -------------------------------------------------------
logret = np.log(nq_1m["close"].astype("float64")).diff()
print(f"\n/NQ continuous front-month, 1m OHLCV")
print(f"  bars      : {len(nq_1m):,}")
print(f"  coverage  : {nq_1m.index[0]}  ->  {nq_1m.index[-1]}")
print(f"  columns   : {list(nq_1m.columns)}")
print(f"  close     : {nq_1m['close'].min():,.2f}  -  {nq_1m['close'].max():,.2f}")
print(f"  duplicate timestamps : {int(nq_1m.index.duplicated().sum())}")
ohlc_bad = int((~((nq_1m["high"] >= nq_1m[["open", "close"]].max(axis=1) - 1e-6) &
                  (nq_1m["low"] <= nq_1m[["open", "close"]].min(axis=1) + 1e-6))).sum())
print(f"  OHLC out of range    : {ohlc_bad}")
print(f"  max |1m log return|  : {logret.abs().max():.4f}"
      "   (roll gaps are NOT price-adjusted -- see note below)")
print(nq_1m.head())

# ---- chart ------------------------------------------------------------------
# 5.3M 1-minute points cannot be rendered legibly, so the CHART aggregates to
# daily bars. `nq_1m` itself keeps every 1-minute bar.
nq_daily = nq_1m.resample("1D").agg(
    {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}
).dropna(subset=["close"])
plot_ohlcv(nq_daily, "/NQ  local front-month (1m source)", "NQ-2010.06.06-2026.01.23",
           interval="1m data, charted daily", bar_minutes=60*24, downsampled=True)

In [ ]:
# cellblock.2

# /NQ yfinance data (1h) - full OHLCV, maximum available history
nq_1h = load_plot_1h("NQ=F", "/NQ")
nq_1h.head()

In [ ]:
# cellblock.3

# /ES yfinance data (1h) - full OHLCV, maximum available history
es_1h = load_plot_1h("ES=F", "/ES")
es_1h.head()

In [ ]:
# cellblock.4

# /RTY yfinance data (1h) - full OHLCV, maximum available history
rty_1h = load_plot_1h("RTY=F", "/RTY")
rty_1h.head()

In [ ]:
# cellblock.5

# /YM yfinance data (1h) - full OHLCV, maximum available history
ym_1h = load_plot_1h("YM=F", "/YM")
ym_1h.head()

In [ ]:
# cellblock.6

# BTC yfinance data (1h) - full OHLCV, maximum available history
btc_1h = load_plot_1h("BTC-USD", "BTC")
btc_1h.head()

In [ ]:
# cellblock.7

# ETH yfinance data (1h) - full OHLCV, maximum available history
eth_1h = load_plot_1h("ETH-USD", "ETH")
eth_1h.head()

In [ ]:
# cellblock.8

# TSLA yfinance data (1h) - full OHLCV, maximum available history
tsla_1h = load_plot_1h("TSLA", "TSLA")
tsla_1h.head()

In [ ]:
# cellblock.9

# NVDA yfinance data (1h) - full OHLCV, maximum available history
nvda_1h = load_plot_1h("NVDA", "NVDA")
nvda_1h.head()

In [ ]:
# cellblock.10

# AAPL yfinance data (1h) - full OHLCV, maximum available history
aapl_1h = load_plot_1h("AAPL", "AAPL")
aapl_1h.head()

In [ ]:
# cellblock.11

# VIX yfinance data (1h) - full OHLCV, maximum available history
vix_1h = load_plot_1h("^VIX", "VIX")
vix_1h.head()

## Module.3
### Re-sampling

Builds every timeframe the strategy needs from the **finest real source available**, plus
maximum-history daily bars for long-horizon work.

| cellblock | builds |
|---|---|
| 1 | resampling engine - `resample_ohlcv()`, `describe_bars()` |
| 2 | fetchers - `fetch_daily()`, `fetch_1m()`, asset registry |
| 3 | `{a}_1d` - daily, maximum history, all 10 assets |
| 4 | `{a}_1m` - 1m bases (/NQ local CSV; others yfinance, chunked) |
| 5 | `{a}_3m` | 
| 6 | `{a}_5m` |
| 7 | `{a}_15m` |
| 8 | `{a}_30m` |
| 9 | `{a}_60m` |
| 10 | `DATASETS` registry + integrity / conservation checks |

Assets: **/NQ, /ES, /RTY, /YM, BTC, ETH, TSLA, NVDA, AAPL, VIX**

---

> **Why the intraday sets are NOT built from the daily bars.**
> Resampling only runs coarse-ward: `1m -> 15m` aggregates trades that actually printed.
> Going `1d -> 1m` is *upsampling* - it would have to invent ~1,380 intrabar prices per day
> that never existed. A backtest fed synthetic intrabar data fills orders at prices nobody
> could have transacted at, which silently manufactures edge. So daily is fetched and kept
> as its own long-history set, and the 1m..60m sets are aggregated **up** from 1-minute bars.
> `resample_ohlcv()` raises on any attempt to upsample.

> **Yahoo's hard limits** (probed, not assumed): `1m` = 8 days per request and ~30 days
> total; `5m/15m/30m` = 60 days; `1h` = 730 days; `1d` = full history. **`3m` is not a
> valid Yahoo interval at all** - it can only ever come from resampling 1m.

> **Known source defect:** Yahoo's *daily* futures bars include some where the open or
> close falls outside that bar's high/low range - /YM 29 bars (0.47%, worst 450 pts, as
> recent as 2025-03-19), /NQ 24, /ES 10; /RTY and the equities are clean. `fetch_daily`
> prints a warning but does **not** repair them, because silently rewriting prices is
> worse than knowing they are wrong. Filter them before using daily bars for fills.

> **`{a}_15m` is rebuilt here** from the 1m source and supersedes the native 15m pull from
> Module.2. For /NQ this is a large upgrade: 366k bars back to 2010 instead of Yahoo's
> 60-day window.

> **Bar convention:** bars are left-labelled (stamped at the bar's *open*), matching both the
> databento CSV and Yahoo. A bar stamped `09:30` on the 15m set covers `[09:30, 09:45)` and
> is therefore only complete at `09:45` - **do not let a signal read it at 09:30.**


In [ ]:
# cellblock.1

# coin flip strategy  -  signal generation only
#     heads -> long  (+1)
#     tails -> short (-1)
# This cellblock decides WHAT the position should be and WHEN that decision is
# allowed to become one. Sizing, costs, fills and P&L are NOT here.
#
# A fair coin has ZERO expected edge before costs and a strictly negative one
# after them. That is the point: this is the null hypothesis the rest of the
# system has to beat. If a backtest ever shows the raw flip making money, the
# backtest is broken - not the coin.
import hashlib

FLIP_SEED    = 20260918          # master seed - change it for a whole new history
HEADS, TAILS = "H", "T"

# Bar-step table, defined here so this cellblock runs standalone - it must not
# depend on any other cellblock in this module having executed first.
TIMEFRAMES = {"1m": "1min", "3m": "3min", "5m": "5min",
              "15m": "15min", "30m": "30min", "60m": "60min"}
BAR_STEPS  = {**TIMEFRAMES, "1d": "1D"}


def _bar_step(df, tf=None):
    """How long one bar lasts. A left-labelled bar is only complete this much later."""
    tf = tf or df.attrs.get("tf")
    if tf is not None:
        return pd.Timedelta(BAR_STEPS.get(tf, tf))
    if len(df) < 3:
        raise ValueError("cannot infer the bar step from < 3 bars - pass tf=")
    return pd.Timedelta(df.index.to_series().diff().median())


def _splitmix64(x):
    """64-bit avalanche. Same answer on every machine, every run, forever."""
    u = np.uint64
    with np.errstate(over="ignore"):
        x = x + u(0x9E3779B97F4A7C15)
        x = (x ^ (x >> u(30))) * u(0xBF58476D1CE4E5B9)
        x = (x ^ (x >> u(27))) * u(0x94D049BB133111EB)
        return x ^ (x >> u(31))


def _stream_key(label, seed):
    """Per-asset key. blake2b, not hash() - hash() is salted per interpreter run."""
    digest = hashlib.blake2b(str(label).encode(), digest_size=8,
                             person=b"coinflip").digest()
    with np.errstate(over="ignore"):
        return np.uint64(int.from_bytes(digest, "big")) ^ np.uint64(seed % 2**64)


def coin_flip_signals(df, label="", tf=None, seed=FLIP_SEED, p_heads=0.5,
                      stream="hash"):
    """Flip one coin per bar:  heads -> long (+1),  tails -> short (-1).

    Returns a frame on the SAME index as `df`:
        flip        H / T
        signal      +1 / -1  - the decision this bar produces
        valid_from  the first instant that decision may be acted on
        position    +1 / -1  - what is actually HELD during this bar

    `signal` and `position` are two columns on purpose. Bars are left-labelled,
    so the bar stamped 09:30 is not finished until 09:45; acting on its own
    signal at 09:30 trades on a bar that has not happened yet. `position` is the
    shifted, tradable version - downstream P&L must use `position`, never
    `signal`. The first `position` is NaN because no decision exists yet.

    stream="hash" (default): a bar's flip is a pure function of (seed, label, bar
    timestamp). Truncating, extending or reordering the sample cannot change any
    bar's flip, and a live session recomputes the same flip for the same bar
    without replaying history - so backtest, paper and live agree by construction.
    stream="sequential" draws one RNG stream in index order: reproducible, but
    every flip shifts if the start of the sample moves.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{label}: index must be a DatetimeIndex, "
                        f"got {type(df.index).__name__}")
    if not df.index.is_monotonic_increasing:
        raise ValueError(f"{label}: index must be sorted ascending")
    if df.index.duplicated().any():
        raise ValueError(f"{label}: {int(df.index.duplicated().sum())} duplicate "
                         "timestamps - one bar cannot get two flips")
    if not 0.0 < p_heads < 1.0:
        raise ValueError(f"{label}: p_heads must be in (0, 1), got {p_heads}")

    step = _bar_step(df, tf)

    if stream == "hash":
        with np.errstate(over="ignore"):
            bits = _splitmix64(df.index.asi8.astype(np.uint64)
                               ^ _stream_key(label, seed))
        draw = (bits >> np.uint64(11)).astype(np.float64) * 2.0**-53
    elif stream == "sequential":
        draw = np.random.default_rng([int(_stream_key(label, seed)),
                                      int(seed % 2**64)]).random(len(df))
    else:
        raise ValueError(f"unknown stream {stream!r} - use 'hash' or 'sequential'")

    heads = draw < p_heads

    out = pd.DataFrame(index=df.index.copy())
    out["flip"]       = pd.Categorical(np.where(heads, HEADS, TAILS),
                                       categories=[HEADS, TAILS])
    out["signal"]     = np.where(heads, 1, -1).astype("int8")
    out["valid_from"] = df.index + step
    out["position"]   = out["signal"].shift(1).astype("float64")
    out.index.name    = "ts"
    out.attrs.update(label=label, tf=tf or df.attrs.get("tf"), seed=seed,
                     p_heads=p_heads, stream=stream, bar_step=step)
    return out


def describe_signals(sig, label=None, tf=None):
    """Coverage + balance + turnover report for one signal set. Returns it unchanged."""
    label = label or sig.attrs.get("label", "")
    tf    = tf    or sig.attrs.get("tf") or ""
    n     = len(sig)
    nh    = int((sig["signal"] == 1).sum())
    p     = sig.attrs.get("p_heads", 0.5)
    # how many standard errors the realised heads rate sits from the coin's own p
    z     = (nh / n - p) / np.sqrt(p * (1 - p) / n) if n else float("nan")
    runs  = sig["signal"].ne(sig["signal"].shift()).cumsum().value_counts()
    held  = sig["position"].dropna()
    turns = int((held != held.shift()).iloc[1:].sum()) if len(held) > 1 else 0

    print(f"{label}  {tf}  coin flip (seed={sig.attrs.get('seed')}, "
          f"stream={sig.attrs.get('stream')})")
    print(f"  bars      : {n:,}")
    print(f"  coverage  : {sig.index[0]}  ->  {sig.index[-1]}")
    print(f"  flips     : {nh:,} heads / {n - nh:,} tails   "
          f"({100 * nh / n:.2f}% long, z={z:+.2f} vs p={p})")
    print(f"  runs      : {len(runs):,} same-way streaks, mean {runs.mean():.2f} bars, "
          f"longest {runs.max()}")
    print(f"  turnover  : {turns:,} position changes "
          f"({100 * turns / max(len(held) - 1, 1):.1f}% of bars, "
          f"~{2 * turns:,} contract-sides to trade before costs)")
    print(f"  tradable  : {int(sig['position'].notna().sum()):,} bars hold a position, "
          f"{int(sig['position'].isna().sum())} flat (no decision yet)")
    return sig


def check_signals(n=5_000, tf="15m", label="SELFTEST", seed=FLIP_SEED):
    """Falsifiable self-test on synthetic bars. One line per property, raises on failure.

    Test 3 is the one that matters: a coin flip reads no market data, so it
    CANNOT peek at the future. The test proves that claim rather than asserting
    it - rewrite every price and the flips must not move.
    """
    idx  = pd.date_range("2024-01-02 09:30", periods=n, freq=TIMEFRAMES[tf],
                         tz="America/New_York")
    rng  = np.random.default_rng(0)
    px   = 100 * np.exp(np.cumsum(rng.normal(0, 1e-3, n)))
    bars = pd.DataFrame({"open": px, "high": px * 1.001, "low": px * 0.999,
                         "close": px, "volume": rng.integers(1, 1e4, n)}, index=idx)
    bars.attrs["tf"] = tf

    sig    = coin_flip_signals(bars, label, tf=tf, seed=seed)
    checks = []

    # 1. position is the signal shifted one bar - the whole no-lookahead contract
    checks.append(("position == signal.shift(1), first bar flat",
                   bool((sig["position"].iloc[1:].to_numpy()
                         == sig["signal"].iloc[:-1].to_numpy()).all())
                   and bool(np.isnan(sig["position"].iloc[0]))))

    # 2. a decision is actionable no later than the bar it is applied in
    checks.append(("valid_from is after its own bar, at or before the next",
                   bool((sig["valid_from"] > sig.index).all())
                   and bool((sig["valid_from"].iloc[:-1].to_numpy()
                             <= sig.index[1:].to_numpy()).all())))

    # 3. flips do not depend on prices at all  ->  look-ahead is impossible
    scrambled = bars.copy()
    scrambled[["open", "high", "low", "close", "volume"]] *= rng.uniform(
        0.5, 2.0, size=(n, 5))
    checks.append(("flips unchanged when every price is rewritten",
                   bool((coin_flip_signals(scrambled, label, tf=tf, seed=seed)["signal"]
                         == sig["signal"]).all())))

    # 4. same seed -> same history, whatever the global RNG has been doing
    np.random.seed(1234)
    checks.append(("reproducible across calls",
                   bool((coin_flip_signals(bars, label, tf=tf, seed=seed)["signal"]
                         == sig["signal"]).all())))

    # 5. hash stream: a bar's flip does not move when the sample window moves
    k = n // 3
    checks.append(("window-invariant (hash stream)",
                   bool((coin_flip_signals(bars.iloc[k:], label, tf=tf, seed=seed)["signal"]
                         .to_numpy() == sig["signal"].iloc[k:].to_numpy()).all())))

    # 6. two assets must not share one coin, or they would trade in lockstep
    checks.append(("independent stream per label",
                   0.40 < float((coin_flip_signals(bars, "OTHER", tf=tf, seed=seed)["signal"]
                                 == sig["signal"]).mean()) < 0.60))

    # 7. a fair coin lands fair - |z| > 4 on 5k flips means a broken generator
    z = ((sig["signal"] == 1).mean() - 0.5) / np.sqrt(0.25 / n)
    checks.append((f"fair coin (z={z:+.2f}, |z| < 4)", abs(z) < 4))

    print(f"coin_flip_signals self-test  ({n:,} synthetic {tf} bars)")
    for name, ok in checks:
        print(f"  [{'OK' if ok else '!!'}] {name}")
    failed = [name for name, ok in checks if not ok]
    if failed:
        raise AssertionError(f"coin flip self-test FAILED: {failed}")
    print("  all properties hold - signals are reproducible and cannot see the future")
    return sig


_ = check_signals()

# Usage, once the bar sets exist:
#   nq_sig = coin_flip_signals(nq_15m, "/NQ", tf="15m")
#   describe_signals(nq_sig)
# Downstream P&L must read `position`, never `signal`.


In [ ]:
# cellblock.2

# entry / exit / risk  -  turns cellblock.1 flips into bounded round trips
#     size  : ALWAYS 1 unit - 1 E-mini contract, 1 share, or 1 coin
#     entry : one hour before the cash open, at that bar's open
#             heads -> long (+1),  tails -> short (-1)
#     stop  : placed exactly RISK_PCT of the account loss limit away
#             ($45,000 x 2% = $900), so one unit losing its stop costs $900
#     exit  : whichever comes first - the stop, or the cash close.
#             Nothing is ever carried overnight.
#
# Size is fixed, so the usual relationship inverts: position sizing does NOT
# absorb risk any more, the STOP DISTANCE does. One unit of a $20/point future
# hits $900 in 45 points; one share of a $250 stock needs a $900 move. The same
# rule therefore means something completely different per asset, and
# describe_trades() prints the stop as a % of price and as a fraction of ATR so
# that difference is impossible to miss.
#
# Read this before trusting any number that comes out of it:
#   - A stop is not a guarantee. It caps the loss only if the market trades
#     through it. Gap past it and the fill is the gap price, which is worse.
#     `planned_risk` is the intent; `worst` in the report is what happened.
#   - OHLC bars cannot say whether the low or the high came first inside a bar.
#     When a bar could have hit the stop, this code assumes it did. That is the
#     pessimistic reading and the only honest one at this resolution.
import math

# ---- account ---------------------------------------------------------------
# `loss_limit` is the number that ends the account - a prop firm's max loss, or
# the balance you refuse to go below. Every risk figure is a fraction OF THAT.
ACCOUNT = {
    "loss_limit":    45_000.0,   # the drawdown that blows the account
    "risk_pct":          0.02,   # <= 2% of it on any one trade  ->  $900
    "daily_loss_pct":    0.06,   # stop for the day after ~3 full stop-outs ($2,700)
}

UNIT_QTY = 1        # one mini contract / share / coin. Not a tunable.

# ---- instruments -----------------------------------------------------------
# point_value = dollars per 1.00 of price move, for ONE unit.
# fee = dollars per side per unit (futures). fee_bps = basis points of notional
# per side (crypto venues charge this way, and on a $90 stop it dominates).
# Verify every one of these against YOUR broker before believing any P&L.
INSTRUMENTS = {
    "/NQ":  dict(unit="1 E-mini contract", point_value=20.0, tick=0.25, fee=2.50, fee_bps=0.0,  tradable=True),
    "/ES":  dict(unit="1 E-mini contract", point_value=50.0, tick=0.25, fee=2.50, fee_bps=0.0,  tradable=True),
    "/RTY": dict(unit="1 E-mini contract", point_value=50.0, tick=0.10, fee=2.50, fee_bps=0.0,  tradable=True),
    "/YM":  dict(unit="1 E-mini contract", point_value=5.0,  tick=1.00, fee=2.50, fee_bps=0.0,  tradable=True),
    "BTC":  dict(unit="1 coin",            point_value=1.0,  tick=0.01, fee=0.0,  fee_bps=10.0, tradable=True),
    "ETH":  dict(unit="1 coin",            point_value=1.0,  tick=0.01, fee=0.0,  fee_bps=10.0, tradable=True),
    "TSLA": dict(unit="1 share",           point_value=1.0,  tick=0.01, fee=0.0,  fee_bps=0.0,  tradable=True),
    "NVDA": dict(unit="1 share",           point_value=1.0,  tick=0.01, fee=0.0,  fee_bps=0.0,  tradable=True),
    "AAPL": dict(unit="1 share",           point_value=1.0,  tick=0.01, fee=0.0,  fee_bps=0.0,  tradable=True),
    "VIX":  dict(unit="index - NOT tradable", point_value=1.0, tick=0.01, fee=0.0, fee_bps=0.0, tradable=False),
    # You cannot hold the VIX index. VIXY is the ETF that tracks short-term VIX
    # futures and IS a share you can buy one of - so VIX stays here as the
    # untradable reference, and VIXY is what the strategy actually trades.
    "VIXY": dict(unit="1 share",           point_value=1.0,  tick=0.01, fee=0.0,  fee_bps=0.0,  tradable=True),
}

# ---- session ---------------------------------------------------------------
# "one hour before market open" = cash_open - entry_lead, in exchange local time.
SESSION = {
    "tz":         "America/New_York",
    "cash_open":  "09:30",
    "cash_close": "16:00",
    "entry_lead": pd.Timedelta("1h"),
}


def risk_per_trade(account=None):
    """Dollars allowed to be lost on one trade. The 2% cap, in money."""
    a = {**ACCOUNT, **(account or {})}
    return a["loss_limit"] * a["risk_pct"]


def stop_distance(symbol, account=None):
    """How far the stop sits, in points, for ONE unit.

    Size is fixed at 1, so this is the only lever left: distance = budget /
    point_value. Rounded DOWN to a whole tick, because rounding up would widen
    the stop past the 2% cap that the whole rule exists to enforce.
    """
    spec = INSTRUMENTS[symbol]
    raw  = risk_per_trade(account) / spec["point_value"]
    return math.floor(raw / spec["tick"]) * spec["tick"]


def wilder_atr(bars, n=14):
    """Wilder's ATR in points. Value at bar t INCLUDES bar t - shift before use.

    Not used to place the stop - the stop comes from the risk budget. This is
    here so the report can say how the stop compares with normal bar movement.
    """
    pc = bars["close"].shift(1)
    tr = pd.concat([bars["high"] - bars["low"],
                    (bars["high"] - pc).abs(),
                    (bars["low"]  - pc).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1.0 / n, adjust=False, min_periods=n).mean()


def round_turn_fees(spec, entry_px, exit_px, qty=UNIT_QTY):
    """Commission both sides, plus notional-based venue fees where they apply."""
    flat = 2.0 * spec["fee"] * qty
    bps  = (abs(entry_px) + abs(exit_px)) * qty * spec["point_value"] \
           * spec["fee_bps"] / 10_000.0
    return flat + bps


class RiskBook:
    """Account-level guard rails. The kill switch lives here.

    Three layers, checked before every entry:
      per trade : the stop is placed at exactly the 2% budget
      per day   : daily_loss_pct of the loss limit, then no more trades today
      account   : drawdown from peak equity >= loss_limit  ->  KILLED, forever
    """

    def __init__(self, account=None):
        a = {**ACCOUNT, **(account or {})}
        self.loss_limit = a["loss_limit"]
        self.daily_cap  = a["loss_limit"] * a["daily_loss_pct"]
        self.equity = self.peak = self.day_pnl = 0.0
        self.day = None
        self.killed, self.kill_ts = False, None

    def roll_to(self, day):
        """New session -> the daily loss budget resets."""
        if day != self.day:
            self.day, self.day_pnl = day, 0.0

    def blocked(self):
        """Why trading is not allowed right now, or None."""
        if self.killed:
            return "kill switch (account loss limit)"
        if self.day_pnl <= -self.daily_cap:
            return "daily loss limit"
        return None

    def record(self, pnl, ts):
        """Book a closed trade and re-check the kill switch."""
        self.equity  += pnl
        self.day_pnl += pnl
        self.peak = max(self.peak, self.equity)
        if not self.killed and (self.peak - self.equity) >= self.loss_limit:
            self.killed, self.kill_ts = True, ts


def run_session_trades(bars, sig, symbol, account=None, session=None,
                       slip_ticks=1.0, allow_untradable=False):
    """One unit, one trade per session: in an hour before the open, out by the close.

    Returns (trades, skips, book).

    No-lookahead, specifically:
      side      comes from sig["position"], which cellblock.1 already shifted -
                it is the flip of a bar that had CLOSED before this entry.
      entry_px  is the entry bar's OPEN, the first price of that bar.
      stop      is a constant derived from the account, not from any bar, so
                nothing inside the entry bar can move it.
    """
    account = {**ACCOUNT, **(account or {})}
    session = {**SESSION, **(session or {})}
    if symbol not in INSTRUMENTS:
        raise KeyError(f"{symbol} not in INSTRUMENTS - add its point value first")
    spec = INSTRUMENTS[symbol]
    if not spec["tradable"] and not allow_untradable:
        raise ValueError(f"{symbol} is {spec['unit']} - you cannot hold a unit of it. "
                         "Trade a future or ETF on it instead, or pass "
                         "allow_untradable=True to model it anyway.")
    if not bars.index.equals(sig.index):
        raise ValueError("bars and signals must share one index - build the "
                         "signals from these exact bars")
    if bars.index.tz is None:
        # Never localize silently. The session rules are wall-clock, so guessing
        # the zone would move every entry by whole hours and the backtest would
        # still "work" - it would just be trading the wrong bar.
        raise ValueError(
            f"{symbol}: bar index is tz-naive, so it cannot be placed on a "
            f"{session['tz']} trading session. yfinance returns tz-naive stamps "
            "for DAILY bars and tz-aware ones for intraday, so a 1d set lands "
            "here naive. Fix it at the source, or localize deliberately:\n"
            '    bars.index = bars.index.tz_localize("America/New_York")\n'
            "and only if you know that is the zone the stamps are already in.")

    budget      = risk_per_trade(account)
    stop_points = stop_distance(symbol, account)
    slip        = slip_ticks * spec["tick"]
    pv          = spec["point_value"]
    tz          = session["tz"]

    et   = bars.index.tz_convert(tz)
    o, h = bars["open"].to_numpy(), bars["high"].to_numpy()
    l, c = bars["low"].to_numpy(),  bars["close"].to_numpy()
    pos  = sig["position"].to_numpy()
    atr  = wilder_atr(bars).shift(1).to_numpy()          # context only

    book, trades, skips = RiskBook(account), [], []

    # Session bounds by binary search on the sorted index, and the stop scan
    # vectorised inside the session. The naive form rescans every bar for every
    # day - fine at 1h, hopeless on a 5M-bar 1m series (hours vs seconds).
    for _midnight in et.normalize().unique():
        day      = _midnight.date()
        open_ts  = pd.Timestamp(f"{day} {session['cash_open']}",  tz=tz)
        close_ts = pd.Timestamp(f"{day} {session['cash_close']}", tz=tz)
        target   = open_ts - session["entry_lead"]
        book.roll_to(day)

        # the entry bar: first bar at or after the target, still before the open
        lo = int(et.searchsorted(target,  "left"))
        hi = int(et.searchsorted(open_ts, "left"))
        if hi <= lo:
            skips.append(dict(day=day, reason="no bar in the hour before the open"))
            continue
        i = lo

        why = book.blocked()
        if why:
            skips.append(dict(day=day, reason=why))
            continue
        if not np.isfinite(pos[i]) or pos[i] == 0:
            skips.append(dict(day=day, reason="no tradable flip yet"))
            continue

        side     = int(pos[i])
        entry_px = o[i] + side * slip                  # pay up to get in
        stop_px  = entry_px - side * stop_points

        # exit scan - the entry bar counts, we are in it from its open
        end = max(int(et.searchsorted(close_ts, "right")), i + 1)
        hit = (l[i:end] <= stop_px) if side > 0 else (h[i:end] >= stop_px)
        if hit.any():
            exit_i  = i + int(hit.argmax())
            # gapped through? then the fill is the gap, not the stop
            exit_px = min(o[exit_i], stop_px) if side > 0 else max(o[exit_i], stop_px)
            reason  = "stop"
        else:
            exit_i, exit_px, reason = end - 1, c[end - 1], "cash close"
        exit_px -= side * slip                         # give up edge to get out

        gross = side * (exit_px - entry_px) * pv * UNIT_QTY
        fees  = round_turn_fees(spec, entry_px, exit_px)
        net   = gross - fees
        book.record(net, bars.index[exit_i])

        trades.append(dict(
            day=day, entry_ts=bars.index[i], exit_ts=bars.index[exit_i],
            side=side, flip="H" if side > 0 else "T", qty=UNIT_QTY,
            entry_px=entry_px, stop_px=stop_px, exit_px=exit_px,
            stop_points=stop_points, planned_risk=stop_points * pv * UNIT_QTY,
            atr=atr[i], exit_reason=reason,
            gross=gross, fees=fees, net=net, equity=book.equity))

    tr = pd.DataFrame(trades)
    if len(tr):
        tr = tr.set_index("entry_ts")
    sk = pd.DataFrame(skips)
    tr.attrs.update(symbol=symbol, budget=budget, stop_points=stop_points,
                    account=account, session=session, slip_ticks=slip_ticks)
    return tr, sk, book


def describe_trades(trades, skips, book, symbol, account=None):
    """Did the risk rules hold, and what did the flip actually cost? Returns trades."""
    a      = {**ACCOUNT, **(account or {})}
    budget = risk_per_trade(a)
    spec   = INSTRUMENTS[symbol]
    pts    = stop_distance(symbol, a)
    n      = len(trades)

    print(f"{symbol}  {spec['unit']}  -  ${budget:,.0f} risk "
          f"({a['risk_pct']:.0%} of ${a['loss_limit']:,.0f})  ->  stop {pts:,.2f} pts")
    if n == 0:
        print("  trades    : NONE")
        if len(skips):
            for why, k in skips["reason"].value_counts().items():
                print(f"      {k:>5}  {why}")
        return trades

    wins  = trades["net"] > 0
    dd    = (trades["equity"].cummax() - trades["equity"]).max()
    worst = trades["net"].min()
    pct   = 100 * pts / trades["entry_px"].mean()
    xatr  = pts / trades["atr"].mean() if trades["atr"].notna().any() else float("nan")

    print(f"  trades    : {n:,}  ({int(wins.sum())} up / {int((~wins).sum())} down, "
          f"{100 * wins.mean():.1f}% win rate)")
    print(f"  stop      : {pts:,.2f} pts = {pct:.3f}% of price = {xatr:.2f}x ATR(14)"
          + ("   [!] inside one bar's normal range - noise will hit it" if xatr < 1
             else "   [!] wider than any plausible day - it will rarely trigger"
             if xatr > 10 else ""))
    print(f"  exits     : " + ", ".join(
        f"{k} {v}" for k, v in trades["exit_reason"].value_counts().items()))
    print(f"  gross     : ${trades['gross'].sum():>12,.2f}")
    print(f"  fees      : ${-trades['fees'].sum():>12,.2f}   "
          f"({trades['fees'].mean():,.2f}/trade = "
          f"{100 * trades['fees'].mean() / budget:.1f}% of the risk budget)")
    print(f"  NET       : ${trades['net'].sum():>12,.2f}   "
          f"({trades['net'].mean():+,.2f}/trade)")
    print(f"  max DD    : ${dd:,.2f} of the ${a['loss_limit']:,.0f} limit "
          f"({100 * dd / a['loss_limit']:.1f}%)")
    breach = trades["planned_risk"] > budget + 1e-9
    print(f"  risk cap  : planned risk <= ${budget:,.0f} on {n - int(breach.sum()):,}/{n:,}"
          + ("  [OK]" if not breach.any() else f"  [!] {int(breach.sum())} BREACH"))
    print(f"  worst     : ${worst:,.2f} realised vs ${budget:,.0f} planned"
          + ("   [!] a gap beat the stop" if worst < -(budget + trades['fees'].max())
             else "   (no gap beat the stop)"))
    print(f"  kill sw   : " + (f"TRIPPED {book.kill_ts}" if book.killed else "not tripped"))
    if len(skips):
        print(f"  skipped   : {len(skips):,} sessions")
        for why, k in skips["reason"].value_counts().head(4).items():
            print(f"      {k:>5}  {why}")
    return trades


def run_all(datasets, account=None, verbose=True, **kw):
    """Apply the same rules to every asset. One unit each. Returns (summary, book_of_trades).

    `datasets` is {label: bars}. Untradable labels (VIX) are reported, not run.
    """
    rows, out = [], {}
    for label, bars in datasets.items():
        spec = INSTRUMENTS.get(label)
        if spec is None:
            rows.append(dict(asset=label, unit="?", trades=0, note="not in INSTRUMENTS"))
            continue
        if not spec["tradable"]:
            rows.append(dict(asset=label, unit=spec["unit"], trades=0,
                             note="cannot hold a unit - skipped"))
            continue
        sig = coin_flip_signals(bars, label, tf=bars.attrs.get("tf"))
        tr, sk, book = run_session_trades(bars, sig, label, account=account, **kw)
        out[label] = (tr, sk, book)
        if verbose:
            describe_trades(tr, sk, book, label, account)
            print()
        if len(tr) == 0:
            note = sk["reason"].mode()[0] if len(sk) else "no sessions"
            rows.append(dict(asset=label, unit=spec["unit"], trades=0, note=note))
            continue
        rows.append(dict(
            asset=label, unit=spec["unit"], trades=len(tr),
            stop_pts=tr["stop_points"].iloc[0],
            stop_pct=100 * tr["stop_points"].iloc[0] / tr["entry_px"].mean(),
            x_atr=tr["stop_points"].iloc[0] / tr["atr"].mean(),
            win_pct=100 * (tr["net"] > 0).mean(),
            stop_pct_exits=100 * (tr["exit_reason"] == "stop").mean(),
            gross=tr["gross"].sum(), fees=-tr["fees"].sum(), net=tr["net"].sum(),
            max_dd=(tr["equity"].cummax() - tr["equity"]).max(),
            killed=book.killed, note=""))
    return pd.DataFrame(rows).set_index("asset"), out


def check_entries_exits(days=90, symbol="/NQ", seed=FLIP_SEED):
    """Falsifiable self-test. One line per rule, raises if any rule is broken."""
    tz  = SESSION["tz"]
    idx = pd.date_range("2024-01-01 00:00", periods=days * 24, freq="60min", tz=tz)
    idx = idx[idx.dayofweek < 5]
    rng = np.random.default_rng(7)
    px  = 18_000 * np.exp(np.cumsum(rng.normal(0, 1.5e-3, len(idx))))
    rad = np.abs(rng.normal(0, 12, len(idx))) + 3
    bars = pd.DataFrame({"open": px, "high": px + rad, "low": px - rad,
                         "close": px + rng.normal(0, 6, len(idx)),
                         "volume": rng.integers(1, 9999, len(idx))}, index=idx)
    bars["high"] = bars[["open", "high", "close"]].max(axis=1)
    bars["low"]  = bars[["open", "low",  "close"]].min(axis=1)
    bars.attrs["tf"] = "60m"

    sig = coin_flip_signals(bars, symbol, tf="60m", seed=seed)
    tr, sk, book = run_session_trades(bars, sig, symbol)
    if len(tr) == 0:
        raise AssertionError("self-test produced no trades - cannot verify anything")

    budget = risk_per_trade()
    et_in  = tr.index.tz_convert(tz)
    et_out = pd.DatetimeIndex(tr["exit_ts"]).tz_convert(tz)
    opens  = pd.to_datetime([f"{d} {SESSION['cash_open']}"  for d in tr["day"]]).tz_localize(tz)
    closes = pd.to_datetime([f"{d} {SESSION['cash_close']}" for d in tr["day"]]).tz_localize(tz)
    checks = []

    # 1. size is one unit. Never two, never zero, never fractional.
    checks.append(("size is exactly 1 unit on every trade",
                   bool((tr["qty"] == 1).all())))

    # 2. the 2% cap - with size fixed, the stop distance is what enforces it
    checks.append((f"planned risk <= ${budget:,.0f} on every trade",
                   bool((tr["planned_risk"] <= budget + 1e-9).all())))

    # 3. entry lands in [one hour before the open, the open)
    checks.append(("entry in the hour before the cash open",
                   bool(((et_in >= opens - SESSION["entry_lead"]) & (et_in < opens)).all())))

    # 4. nothing is carried overnight
    checks.append(("flat by the cash close, every session",
                   bool((et_out <= closes).all())
                   and bool((pd.Index(et_out.date) == pd.Index(tr["day"])).all())))

    # 5. heads -> long, tails -> short, using the SHIFTED decision
    want = sig["position"].reindex(tr.index).to_numpy()
    checks.append(("side == the flip already on the book (heads long, tails short)",
                   bool((tr["side"].to_numpy() == want).all())
                   and bool((np.where(tr["flip"] == "H", 1, -1) == tr["side"]).all())))

    # 6. nothing inside the entry bar may move the stop  ->  no look-ahead
    poke  = bars.copy()
    first = bars.index.get_loc(tr.index[0])
    poke.iloc[first, poke.columns.get_loc("high")] *= 1.05
    poke.iloc[first, poke.columns.get_loc("low")]  *= 0.95
    tr2, _, _ = run_session_trades(poke, coin_flip_signals(poke, symbol, tf="60m", seed=seed),
                                   symbol)
    checks.append(("stop unmoved when the entry bar's own high/low are rewritten",
                   bool(abs(tr2["stop_px"].iloc[0] - tr["stop_px"].iloc[0]) < 1e-9)))

    # 7. a stop fill is never BETTER than the stop - gaps fill worse, not free
    st = tr[tr["exit_reason"] == "stop"]
    checks.append((f"stop fills never better than the stop ({len(st)} stop-outs)",
                   bool((st["side"] * (st["exit_px"] - st["stop_px"]) <= 1e-9).all())))

    # 8. an untradable instrument must be refused, not silently modelled
    try:
        run_session_trades(bars, sig, "VIX")
        refused = False
    except ValueError:
        refused = True
    checks.append(("refuses to trade a unit of an index (VIX)", refused))

    # 9. once the kill switch trips, the account never trades again
    tiny = {"loss_limit": 300.0, "risk_pct": 0.30, "daily_loss_pct": 10.0}
    tr3, _, bk3 = run_session_trades(bars, sig, symbol, account=tiny)
    checks.append(("kill switch blocks every entry after it trips",
                   (not bk3.killed) or bool((pd.DatetimeIndex(tr3["exit_ts"])
                                             <= bk3.kill_ts).all())))

    print(f"entry / exit / risk self-test  ({len(tr):,} trades on {days} synthetic days)")
    for name, ok in checks:
        print(f"  [{'OK' if ok else '!!'}] {name}")
    failed = [name for name, ok in checks if not ok]
    if failed:
        raise AssertionError(f"risk self-test FAILED: {failed}")
    print("  every entry, exit and risk rule holds on synthetic data")
    return tr


_ = check_entries_exits()

# Usage, once cellblock.1 has made the flips:
#   sig          = coin_flip_signals(nq_1h, "/NQ", tf="60m")
#   tr, sk, book = run_session_trades(nq_1h, sig, "/NQ")
#   describe_trades(tr, sk, book, "/NQ")
#
#   summary, all_trades = run_all({"/NQ": nq_1h, "BTC": btc_1h, "AAPL": aapl_1h})
#   summary


In [ ]:
# cellblock.3

# P&L visuals  -  what the rules in cellblock.2 actually did to the account
#     collect_assets : every {asset}_{tf} frame this notebook has built
#     plot_pnl       : equity + drawdown vs the loss limit, one dataset
#     plot_pnl_fan   : one dataset under COIN_SEEDS coins - the dispersion chart
#     plot_pnl_grid  : small multiples, EVERY dataset, one equity panel each
#     plot_fan_grid  : small multiples, EVERY dataset, COIN_SEEDS coins each
#
# Two rules the grids follow, both about not lying by omission:
#   - A dataset that cannot trade still gets a panel, with the reason printed in
#     it. Dropping it would read as "no result"; "no bar in the hour before the
#     open" reads as "this data cannot answer the question", which is the truth.
#   - Panels do NOT share an x axis, because the datasets do not share a period:
#     /NQ 1m is 15.6 years, the Yahoo 1h sets are ~2. Each panel therefore
#     carries its own date span in the title - compare the shapes, not the
#     widths, and trust the long sample over the short ones.
#
# Every function RETURNS the frame it drew, so each figure has a table behind it.

COIN_SEEDS = 50     # how many coins every dispersion chart draws

# label -> the notebook's variable stem. Timeframes are discovered, not assumed,
# so whatever Module.2/3 has built shows up without editing this cell.
VAR_OF = {"/NQ": "nq", "/ES": "es", "/RTY": "rty", "/YM": "ym",
          "BTC": "btc", "ETH": "eth", "TSLA": "tsla", "NVDA": "nvda",
          "AAPL": "aapl", "VIX": "vix", "VIXY": "vixy"}
# 1d is deliberately absent: the entry rule needs a bar INSIDE the hour before
# the cash open, and a daily bar has no inside. Every 1d set would render as an
# empty panel, so they are excluded here rather than drawn as ten blanks.
TF_ORDER = ["1m", "3m", "5m", "15m", "30m", "60m", "1h"]
GRID_TFS      = ["1m", "1h"]   # what the grids use unless told otherwise
MIN_SPAN_DAYS = 180            # shorter than this is not a sample, it is an anecdote

# "$" is data in every label on these charts, not a LaTeX delimiter. Left on,
# matplotlib reads the text between two "$" as mathtext and silently eats the
# spacing ("$-45,690 to +169,830" renders as "-45,690to+169,830").
plt.rcParams["text.parse_math"] = False

# Chart tokens. Categorical slot 1 (blue) for the series, status-critical for
# loss. Validated as a pair against both surfaces: CVD dE 23.8 light / 25.7 dark,
# normal-vision 31.6 / 31.9, both >= 3:1 on their surface.
VIZ = {
    "light": dict(surface="#fcfcfb", ink="#0b0b0b", secondary="#52514e",
                  muted="#898781", grid="#e1e0d9", axis="#c3c2b7",
                  series="#2a78d6", loss="#d03b3b", ensemble="#898781"),
    "dark":  dict(surface="#1a1a19", ink="#ffffff", secondary="#c3c2b7",
                  muted="#898781", grid="#2c2c2a", axis="#383835",
                  series="#3987e5", loss="#d03b3b", ensemble="#898781"),
}


def collect_assets(scope=None, tfs=GRID_TFS, min_days=MIN_SPAN_DAYS, quiet=False):
    """Every {asset}_{tf} frame in the notebook -> {panel label: (symbol, bars)}.

    Discovers what exists rather than demanding a fixed list, so /NQ's 15.6-year
    local 1m set and the ~2-year Yahoo 1h sets land side by side.

    Two defaults keep the grids readable, and both announce themselves:
      tfs=GRID_TFS  - 1m and 1h only. Once cellblock.4-9 have built every
                      timeframe there are 70 frames in the kernel, and a
                      70-panel grid is a wall, not a chart.
      min_days      - sets spanning less than this are dropped AND NAMED. After
                      cellblock.4 the nine Yahoo 1m bases are ~30 days each;
                      30 days is not evidence about anything.

    For deliberate multi-timeframe work, ask for it:
        collect_assets(tfs=TF_ORDER, min_days=0)
    Daily is absent from TF_ORDER on purpose - see the note there.
    """
    g, out, dropped = (scope if scope is not None else globals()), {}, []
    for symbol, var in VAR_OF.items():
        for tf in tfs:
            df = g.get(f"{var}_{tf}")
            if not (isinstance(df, pd.DataFrame) and len(df)
                    and isinstance(df.index, pd.DatetimeIndex)):
                continue
            df.attrs.setdefault("tf", tf)
            if df.index.tz is None:
                # yfinance hands back tz-naive stamps for DAILY bars. The
                # session rules are wall-clock, so a naive set cannot be placed
                # on a trading day - named here rather than crashing a grid.
                dropped.append(f"{symbol} {tf} (tz-naive)")
                continue
            days = (df.index[-1] - df.index[0]).days
            if days < min_days:
                dropped.append(f"{symbol} {tf} ({days}d)")
                continue
            out[f"{symbol} {tf}"] = (symbol, df)

    if not quiet:
        print(f"collected {len(out)} datasets across "
              f"{len({s for s, _ in out.values()})} assets")
        for lab, (sym, df) in out.items():
            yrs = (df.index[-1] - df.index[0]).days / 365.25
            print(f"  {lab:12} {len(df):>10,} bars  "
                  f"{df.index[0]:%Y-%m-%d} -> {df.index[-1]:%Y-%m-%d}  ({yrs:5.1f} yrs)"
                  + ("" if INSTRUMENTS[sym]["tradable"] else "   [not tradable]"))
        if dropped:
            print(f"  dropped ({min_days}d minimum, tz-aware only): "
                  f"{', '.join(dropped)}")
    return out


def _style(ax, t, ylabel=None, money=True, small=False):
    """Recessive chrome: hairline grid, muted ticks, no box, dollars on y."""
    ax.set_facecolor(t["surface"])
    ax.grid(True, color=t["grid"], linewidth=0.8, alpha=1.0)
    ax.set_axisbelow(True)
    for side, spine in ax.spines.items():
        spine.set_visible(side == "bottom")
        spine.set_color(t["axis"])
        spine.set_linewidth(1.0)
    ax.tick_params(colors=t["muted"], labelsize=7 if small else 8, length=0)
    if money:
        ax.yaxis.set_major_formatter(plt.FuncFormatter(
            lambda v, _: (f"-${abs(v)/1000:,.0f}k" if abs(v) >= 1000 else f"-${abs(v):,.0f}")
            if v < 0 else (f"${v/1000:,.0f}k" if v >= 1000 else f"${v:,.0f}")))
    if ylabel:
        ax.set_ylabel(ylabel, color=t["secondary"], fontsize=9)
    return ax


def _equity_path(trades):
    """Equity stepped through time, anchored at 0 before the first trade."""
    x = [trades.index[0]] + list(pd.DatetimeIndex(trades["exit_ts"]))
    y = [0.0] + list(trades["equity"])
    return pd.DatetimeIndex(x), np.asarray(y, dtype=float)


def _span(df):
    """'2010-2026' - the panel's own period, since panels do not share an axis."""
    return f"{df.index[0]:%Y}-{df.index[-1]:%Y}"


def _blank_panel(ax, t, label, why):
    """A dataset that produced nothing still gets a panel, and a reason."""
    ax.set_facecolor(t["surface"])
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.add_patch(plt.Rectangle((0.02, 0.06), 0.96, 0.88, transform=ax.transAxes,
                               fill=False, edgecolor=t["grid"], linewidth=1.0,
                               linestyle=(0, (4, 4))))
    ax.text(0.5, 0.58, label, ha="center", va="center", fontsize=9.5,
            color=t["secondary"], transform=ax.transAxes)
    ax.text(0.5, 0.36, why, ha="center", va="center", fontsize=7.5,
            color=t["muted"], transform=ax.transAxes)


def _run_one(bars, symbol, account=None, seed=FLIP_SEED, **kw):
    """Run one dataset. Returns (trades, book, reason-it-produced-nothing)."""
    spec = INSTRUMENTS[symbol]
    if not spec["tradable"]:
        return None, None, f"{spec['unit']}\ncannot hold a unit"
    try:
        sig = coin_flip_signals(bars, symbol, tf=bars.attrs.get("tf"), seed=seed)
        tr, sk, bk = run_session_trades(bars, sig, symbol, account=account, **kw)
    except Exception as exc:
        # A grid draws many datasets. One malformed set must degrade to a
        # labelled panel, not take down every other panel with it - the reason
        # is printed in the panel, so nothing is swallowed.
        return None, None, f"{type(exc).__name__}\n{str(exc).splitlines()[0][:64]}"
    if len(tr) == 0:
        return None, None, (sk["reason"].mode()[0] if len(sk) else "no sessions")
    return tr, bk, None


def plot_pnl(trades, book, symbol, account=None, theme="light", figsize=(13, 7)):
    """Equity curve over drawdown, for one dataset. Returns the plotted frame.

    Two stacked panels on ONE shared time axis - never two y-scales on one plot.
    Top: cumulative net P&L. Bottom: drawdown from peak, against the account
    loss limit that ends the run.
    """
    a = {**ACCOUNT, **(account or {})}
    t = VIZ[theme]
    if trades is None or len(trades) == 0:
        print(f"{symbol}: no trades to plot")
        return trades

    x, eq = _equity_path(trades)
    dd    = np.maximum.accumulate(eq) - eq
    limit = a["loss_limit"]
    final = eq[-1]

    fig, (ax1, ax2) = plt.subplots(
        2, 1, sharex=True, figsize=figsize, facecolor=t["surface"],
        gridspec_kw={"height_ratios": [2.6, 1], "hspace": 0.08})

    _style(ax1, t, "Cumulative net P&L ($)")
    ax1.axhline(0, color=t["axis"], linewidth=1.0)
    ax1.plot(x, eq, color=t["series"], linewidth=2.0, solid_joinstyle="round")
    ax1.fill_between(x, 0, eq, color=t["series"], alpha=0.10, linewidth=0)
    # one direct label, on the value that matters - not a number on every point
    ax1.plot([x[-1]], [final], "o", markersize=8, color=t["series"],
             markeredgecolor=t["surface"], markeredgewidth=2, zorder=5)
    ax1.annotate(f"  ${final:+,.0f}", (x[-1], final), color=t["ink"],
                 fontsize=10, fontweight="bold", va="center")
    ax1.set_title(
        f"{symbol}  -  cumulative net P&L, {len(trades):,} coin-flip trades"
        f"   |   {INSTRUMENTS[symbol]['unit']}, ${risk_per_trade(a):,.0f} risk/trade"
        f"   |   {x[0]:%Y-%m-%d} to {x[-1]:%Y-%m-%d}",
        color=t["ink"], fontsize=10.5, loc="left", pad=12)

    _style(ax2, t, "Drawdown ($)")
    ax2.fill_between(x, 0, -dd, color=t["loss"], alpha=0.18, linewidth=0)
    ax2.plot(x, -dd, color=t["loss"], linewidth=1.6)
    ax2.axhline(-limit, color=t["loss"], linewidth=1.2, linestyle=(0, (5, 4)))
    ax2.annotate(f"account loss limit  -${limit:,.0f}", (x[0], -limit),
                 color=t["loss"], fontsize=8.5, va="bottom", ha="left",
                 xytext=(4, 3), textcoords="offset points")
    ax2.set_ylim(min(-limit * 1.12, -dd.max() * 1.12), limit * 0.06)
    _wi   = int(np.argmax(dd))
    _frac = _wi / max(len(x) - 1, 1)
    _ha   = "right" if _frac > 0.72 else "left" if _frac < 0.28 else "center"
    _dx   = -6 if _ha == "right" else 6 if _ha == "left" else 0
    ax2.annotate(f"worst  -${dd.max():,.0f}  ({100 * dd.max() / limit:.0f}% of the limit)",
                 (x[_wi], -dd.max()), color=t["ink"], fontsize=8.5,
                 va="bottom", ha=_ha, xytext=(_dx, 8), textcoords="offset points")

    # kill switch gets a line AND a label - never colour alone
    if book is not None and book.killed:
        for ax in (ax1, ax2):
            ax.axvline(book.kill_ts, color=t["loss"], linewidth=1.2,
                       linestyle=(0, (2, 3)), zorder=1)
        ax1.annotate("  KILL SWITCH TRIPPED", (book.kill_ts, ax1.get_ylim()[1]),
                     color=t["loss"], fontsize=9, fontweight="bold", va="top")

    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    fig.autofmt_xdate()
    plt.show()
    out = pd.DataFrame({"equity": eq, "drawdown": dd}, index=x)
    out.index.name = "ts"
    return out


def _fan_paths(bars, symbol, seeds, base_seed, account, **kw):
    """Run `seeds` coins over one dataset. Returns (paths, deaths, rows)."""
    paths, deaths, rows = [], [], []
    for k in range(seeds):
        seed = base_seed + k
        tr, bk, _why = _run_one(bars, symbol, account=account, seed=seed, **kw)
        if tr is None:
            continue
        x, eq = _equity_path(tr)
        paths.append((x, eq))
        if bk.killed:
            deaths.append((x[-1], eq[-1]))
        rows.append(dict(seed=seed, trades=len(tr), net=eq[-1],
                         max_dd=float((np.maximum.accumulate(eq) - eq).max()),
                         killed=bk.killed))
    return paths, deaths, rows


def plot_pnl_fan(bars, symbol, seeds=COIN_SEEDS, base_seed=FLIP_SEED, account=None,
                 theme="light", figsize=(13, 6.5), **kw):
    """One dataset under `seeds` different coins. Returns the per-seed summary.

    This is the chart that says whether an equity curve means anything. Every
    grey line is the identical strategy on identical bars - only the coin
    differs. The width of that bundle is the honest uncertainty; the blue line
    is just the one draw you happened to look at first.
    """
    a = {**ACCOUNT, **(account or {})}
    t, limit = VIZ[theme], {**ACCOUNT, **(account or {})}["loss_limit"]
    paths, deaths, rows = _fan_paths(bars, symbol, seeds, base_seed, account, **kw)
    if not paths:
        print(f"{symbol}: no trades under any seed")
        return pd.DataFrame()

    summary = pd.DataFrame(rows).set_index("seed")
    fig, ax = plt.subplots(figsize=figsize, facecolor=t["surface"])
    _style(ax, t, "Cumulative net P&L ($)")
    ax.axhline(0, color=t["axis"], linewidth=1.0)
    for x, eq in paths:
        ax.plot(x, eq, color=t["ensemble"], linewidth=0.8, alpha=0.30, zorder=2)
    # the kill switch fires on DRAWDOWN FROM PEAK, not on an absolute balance,
    # so there is no horizontal line to draw for it - a run can die at +$90k.
    # Mark where each one actually ended instead.
    if deaths:
        ax.plot([d[0] for d in deaths], [d[1] for d in deaths], "x", markersize=7,
                markeredgewidth=1.6, color=t["loss"], zorder=6)
    x0, eq0 = paths[0]
    ax.plot(x0, eq0, color=t["series"], linewidth=2.2, zorder=4)
    ax.plot([x0[-1]], [eq0[-1]], "o", markersize=8, color=t["series"],
            markeredgecolor=t["surface"], markeredgewidth=2, zorder=5)

    # legend: identity is never carried by colour alone
    ax.plot([], [], color=t["ensemble"], linewidth=0.8, alpha=0.6,
            label=f"{len(paths)} coin seeds, identical rules")
    ax.plot([], [], color=t["series"], linewidth=2.2,
            label=f"seed {base_seed} (the one in the notebook)")
    if deaths:
        ax.plot([], [], "x", markersize=7, markeredgewidth=1.6, color=t["loss"],
                label=f"kill switch tripped ({len(deaths)} runs ended here)")
    leg = ax.legend(loc="upper left", frameon=False, fontsize=8.5)
    for txt in leg.get_texts():
        txt.set_color(t["secondary"])

    killed = int(summary["killed"].sum())
    ax.set_title(
        f"{symbol}  -  the same strategy under {len(paths)} different coins"
        f"   |   net ${summary['net'].min():+,.0f} to ${summary['net'].max():+,.0f}"
        f",  {killed}/{len(paths)} hit the ${limit:,.0f} drawdown limit",
        color=t["ink"], fontsize=10.5, loc="left", pad=12)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    fig.autofmt_xdate()
    plt.show()
    return summary


def plot_pnl_grid(sets=None, account=None, theme="light", ncols=3,
                  panel=(4.5, 2.7), **kw):
    """ONE equity panel per dataset, every asset in the notebook. Returns a table.

    Ten-plus datasets on one axis would be cycled colours and an unreadable
    tangle, so this facets: one series per panel, no legend needed, every panel
    directly labelled with its own net and its own date span.
    """
    a    = {**ACCOUNT, **(account or {})}
    t    = VIZ[theme]
    sets = collect_assets(quiet=True) if sets is None else sets
    if not sets:
        print("no datasets found - run Module.2 first")
        return pd.DataFrame()

    nrows = math.ceil(len(sets) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel[0] * ncols, panel[1] * nrows),
                             facecolor=t["surface"], squeeze=False)
    rows = []
    for ax, (lab, (sym, bars)) in zip(axes.ravel(), sets.items()):
        tr, bk, why = _run_one(bars, sym, account=account, **kw)
        if tr is None:
            _blank_panel(ax, t, f"{lab}   no trades", why)
            rows.append(dict(dataset=lab, symbol=sym, span=_span(bars), trades=0,
                             net=np.nan, max_dd=np.nan, killed=False, note=why.replace("\n", " ")))
            continue
        x, eq = _equity_path(tr)
        dd    = float((np.maximum.accumulate(eq) - eq).max())
        _style(ax, t, small=True)
        ax.axhline(0, color=t["axis"], linewidth=1.0)
        col = t["series"] if eq[-1] >= 0 else t["loss"]
        ax.plot(x, eq, color=col, linewidth=1.6)
        ax.fill_between(x, 0, eq, color=col, alpha=0.10, linewidth=0)
        ax.set_title(f"{lab}  {_span(bars)}   ${eq[-1]:+,.0f}   {len(tr):,} trades"
                     + ("   KILLED" if bk.killed else ""),
                     color=t["loss"] if bk.killed else t["ink"],
                     fontsize=8.5, loc="left", pad=5)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        rows.append(dict(dataset=lab, symbol=sym, span=_span(bars), trades=len(tr),
                         net=eq[-1], max_dd=dd, killed=bk.killed, note=""))
    for ax in axes.ravel()[len(sets):]:
        ax.set_visible(False)
    fig.suptitle(f"Cumulative net P&L, every dataset in the notebook  -  1 unit each, "
                 f"${risk_per_trade(a):,.0f} risk/trade, one coin (seed {FLIP_SEED})"
                 f"   ·   panels do NOT share an x axis",
                 color=t["ink"], fontsize=11, x=0.006, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.975))
    plt.show()
    return pd.DataFrame(rows).set_index("dataset")


def plot_fan_grid(sets=None, seeds=COIN_SEEDS, base_seed=FLIP_SEED, account=None,
                  theme="light", ncols=3, panel=(4.5, 2.7), **kw):
    """COIN_SEEDS coins on EVERY dataset, one panel each. Returns a table.

    The grid version of the only chart worth trusting. Read the spread inside
    each panel before reading the level of any single line: if the bundle
    straddles zero, that dataset has told you nothing about edge.
    """
    a    = {**ACCOUNT, **(account or {})}
    t    = VIZ[theme]
    sets = collect_assets(quiet=True) if sets is None else sets
    if not sets:
        print("no datasets found - run Module.2 first")
        return pd.DataFrame()
    limit = a["loss_limit"]

    nrows = math.ceil(len(sets) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel[0] * ncols, panel[1] * nrows),
                             facecolor=t["surface"], squeeze=False)
    rows = []
    for ax, (lab, (sym, bars)) in zip(axes.ravel(), sets.items()):
        print(f"  {lab:12} {seeds} coins ...", end="", flush=True)
        paths, deaths, seed_rows = _fan_paths(bars, sym, seeds, base_seed, account, **kw)
        if not paths:
            _, _, why = _run_one(bars, sym, account=account, **kw)
            print(f" none ({why})".replace("\n", " "))
            _blank_panel(ax, t, f"{lab}   no trades", why or "no trades")
            rows.append(dict(dataset=lab, symbol=sym, span=_span(bars), seeds=0,
                             net_min=np.nan, net_med=np.nan, net_max=np.nan,
                             pct_profitable=np.nan, killed=0,
                             note=(why or "").replace("\n", " ")))
            continue
        s = pd.DataFrame(seed_rows)
        print(f" median ${s['net'].median():+,.0f}, {int(s['killed'].sum())} killed")

        _style(ax, t, small=True)
        ax.axhline(0, color=t["axis"], linewidth=1.0)
        for x, eq in paths:
            ax.plot(x, eq, color=t["ensemble"], linewidth=0.7, alpha=0.28, zorder=2)
        if deaths:
            ax.plot([d[0] for d in deaths], [d[1] for d in deaths], "x", markersize=5,
                    markeredgewidth=1.2, color=t["loss"], zorder=6)
        ax.plot(paths[0][0], paths[0][1], color=t["series"], linewidth=1.7, zorder=4)
        killed = int(s["killed"].sum())
        ax.set_title(f"{lab}  {_span(bars)}   median ${s['net'].median():+,.0f}"
                     f"   {killed}/{len(paths)} killed"
                     f"   {100 * (s['net'] > 0).mean():.0f}% profitable",
                     color=t["loss"] if killed > len(paths) / 2 else t["ink"],
                     fontsize=8.5, loc="left", pad=5)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        rows.append(dict(dataset=lab, symbol=sym, span=_span(bars), seeds=len(paths),
                         net_min=s["net"].min(), net_med=s["net"].median(),
                         net_max=s["net"].max(),
                         pct_profitable=100 * (s["net"] > 0).mean(),
                         killed=killed, note=""))
    for ax in axes.ravel()[len(sets):]:
        ax.set_visible(False)

    # one figure-level legend - per-panel legends would be pure noise here
    h = [plt.Line2D([], [], color=t["ensemble"], lw=0.9, alpha=0.6),
         plt.Line2D([], [], color=t["series"], lw=1.7),
         plt.Line2D([], [], color=t["loss"], marker="x", lw=0, markersize=6,
                    markeredgewidth=1.4)]
    leg = fig.legend(h, [f"{seeds} coin seeds, identical rules",
                         f"seed {base_seed}",
                         f"kill switch tripped (${limit:,.0f} drawdown)"],
                     loc="upper right", frameon=False, fontsize=8.5, ncols=3,
                     bbox_to_anchor=(0.995, 1.0))
    for txt in leg.get_texts():
        txt.set_color(t["secondary"])
    fig.suptitle(f"The same coin-flip strategy under {seeds} coins, every dataset"
                 f"  -  1 unit each, ${risk_per_trade(a):,.0f} risk/trade"
                 f"   ·   panels do NOT share an x axis",
                 color=t["ink"], fontsize=11, x=0.006, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 0.972))
    plt.show()
    return pd.DataFrame(rows).set_index("dataset")


def check_pnl_plots(days=120, symbol="/NQ", seeds=COIN_SEEDS):
    """Self-test: the drawn numbers must equal the traded numbers."""
    tz  = SESSION["tz"]
    idx = pd.date_range("2024-01-01 00:00", periods=days * 24, freq="60min", tz=tz)
    idx = idx[idx.dayofweek < 5]
    rng = np.random.default_rng(11)
    px  = 18_000 * np.exp(np.cumsum(rng.normal(0, 1.5e-3, len(idx))))
    rad = np.abs(rng.normal(0, 12, len(idx))) + 3
    bars = pd.DataFrame({"open": px, "high": px + rad, "low": px - rad,
                         "close": px + rng.normal(0, 6, len(idx)),
                         "volume": rng.integers(1, 9999, len(idx))}, index=idx)
    bars["high"] = bars[["open", "high", "close"]].max(axis=1)
    bars["low"]  = bars[["open", "low",  "close"]].min(axis=1)
    bars.attrs["tf"] = "60m"

    sig = coin_flip_signals(bars, symbol, tf="60m")
    tr, sk, bk = run_session_trades(bars, sig, symbol)
    curve = plot_pnl(tr, bk, symbol)
    checks = []

    # 1. the curve's endpoint IS the sum of the trades - no cosmetic smoothing
    checks.append(("equity endpoint == sum of net P&L",
                   abs(curve["equity"].iloc[-1] - tr["net"].sum()) < 1e-6))

    # 2. the curve starts flat at zero, before any trade has closed
    checks.append(("curve anchored at 0 before the first trade",
                   abs(curve["equity"].iloc[0]) < 1e-12
                   and curve.index[0] == tr.index[0]))

    # 3. the drawdown panel matches the drawdown the risk book measured
    checks.append(("plotted drawdown == peak-to-trough of the equity",
                   abs(curve["drawdown"].max()
                       - (tr["equity"].cummax() - tr["equity"]).max()) < 1e-6))

    # 4. every point is monotone in time - nothing is drawn out of order
    checks.append(("time axis strictly non-decreasing",
                   bool(curve.index.is_monotonic_increasing)))

    # 5. the fan really varies the coin over all COIN_SEEDS draws
    fan = plot_pnl_fan(bars, symbol, seeds=seeds)
    checks.append((f"fan draws {seeds} distinct coins with a real spread",
                   len(fan) == seeds and fan["net"].nunique() == seeds))

    # 6. a grid must not silently drop a dataset it cannot trade
    grid = plot_pnl_grid({"SELFTEST 60m": (symbol, bars),
                          "VIX 60m": ("VIX", bars)}, ncols=2)
    checks.append(("untradable dataset still gets a row, with a reason",
                   len(grid) == 2 and grid.loc["VIX 60m", "trades"] == 0
                   and bool(grid.loc["VIX 60m", "note"])))

    print(f"P&L chart self-test  ({len(tr):,} trades, {seeds} coins)")
    for name, ok in checks:
        print(f"  [{'OK' if ok else '!!'}] {name}")
    failed = [name for name, ok in checks if not ok]
    if failed:
        raise AssertionError(f"P&L chart self-test FAILED: {failed}")
    print("  the charts show the trades, not a prettier version of them")
    return curve


_ = check_pnl_plots()

# Usage, once Module.2 has loaded the data:
#   sets = collect_assets()          # /NQ 1m local + every {asset}_1h
#   plot_pnl_grid(sets)              # one coin,  every dataset
#   plot_fan_grid(sets)              # 50 coins,  every dataset   <- read this one
#
#   tr, bk, _ = _run_one(nq_1m, "/NQ")
#   plot_pnl(tr, bk, "/NQ")          # one dataset, full detail


In [ ]:
# cellblock.4

# data layer  +  the 1m bases every resampled set below is aggregated from.
#
# The resampling engine and the fetchers live HERE because the cells that used
# to hold them - Module.3 cellblock.1 and cellblock.2 - are now the strategy and
# the risk layer. cellblocks 5 to 10 below call into what this cellblock
# defines, so this one has to run before any of them.
#
#   /NQ    : LOCAL continuous front-month from Module.2 cellblock.1
#            (5.3M bars, 2010-2026) - NOT Yahoo's 30-day window.
#   others : yfinance 1m, fetched in chunks, reaches back only ~30 days.
#            Anything resampled from those inherits that 30-day ceiling.
#
# It also builds {asset}_1h over YF_PERIOD (720d) for ALL ten assets - the only
# set with real history for TSLA, NVDA, AAPL and VIX, and the one the strategy
# actually trades.
import time
import logging
from contextlib import contextmanager

AGG = {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}

# cellblock.1 already defines TIMEFRAMES. Repeated identically so this cellblock
# still stands up if the strategy cells have not been run in this kernel.
TIMEFRAMES = {"1m": "1min", "3m": "3min", "5m": "5min",
              "15m": "15min", "30m": "30min", "60m": "60min"}

ASSETS = [("nq",   "/NQ",  "NQ=F"),
          ("es",   "/ES",  "ES=F"),
          ("rty",  "/RTY", "RTY=F"),
          ("ym",   "/YM",  "YM=F"),
          ("btc",  "BTC",  "BTC-USD"),
          ("eth",  "ETH",  "ETH-USD"),
          ("tsla", "TSLA", "TSLA"),
          ("nvda", "NVDA", "NVDA"),
          ("aapl", "AAPL", "AAPL"),
          ("vix",  "VIX",  "^VIX"),
          ("vixy", "VIXY", "VIXY")]     # tradable proxy for the VIX index

RESAMPLED = ["3m", "5m", "15m", "30m", "60m"]


def resample_ohlcv(df, tf, src_step=None, drop_partial=True, label=""):
    """Aggregate an OHLCV frame up to timeframe `tf`.

    Bars are left-labelled/left-closed, so a bar is stamped at its OPEN and only
    becomes complete one `tf` later. Empty bins (weekends, halts, session breaks)
    are dropped rather than forward-filled - a forward-filled bar is a bar that
    never traded. Upsampling is refused outright.

    The refusal MEASURES the source step off the index rather than believing
    `src_step`. It used to trust the argument, and every batch cellblock below
    passes the same constant: hand a 1h frame to resample_ohlcv(df, "15m") with
    src_step="1min" and the guard saw 15min > 1min, raised nothing, and returned
    hourly bars wearing a 15m label - one source bar per bin. Silent upsampling
    is exactly the failure this function exists to prevent, so the measurement
    wins, and a declared step can only make the check stricter, never looser.
    """
    freq = TIMEFRAMES.get(tf, tf)
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{label}: index must be a DatetimeIndex, got {type(df.index).__name__}")
    if not df.index.is_monotonic_increasing:
        df = df.sort_index()

    measured = (pd.Timedelta(df.index.to_series().diff().median())
                if len(df) >= 3 else None)
    declared = pd.Timedelta(src_step) if src_step is not None else None
    src = max([s for s in (measured, declared) if s is not None and s > pd.Timedelta(0)],
              default=pd.Timedelta("1min"))
    if pd.Timedelta(freq) < src:
        raise ValueError(
            f"{label}: refusing to resample a {src} source UP to {tf}. "
            "Upsampling fabricates bars that never traded."
            + ("" if declared is None or declared == src else
               f" (caller declared {declared}; the index says {measured})"))

    g     = df.resample(freq, label="left", closed="left")
    out   = g.agg(AGG)
    n_src = g.size().reindex(out.index)
    out, n_src = out[n_src > 0], n_src[n_src > 0]        # drop empty bins

    # a trailing bin the source never reached is incomplete -> its "close" is fake
    dropped_partial = False
    if drop_partial and len(out):
        if df.index[-1] + src < out.index[-1] + pd.Timedelta(freq):
            out, n_src, dropped_partial = out.iloc[:-1], n_src.iloc[:-1], True

    out["volume"] = out["volume"].fillna(0)
    out.index.name = "ts"
    # store the MEASURED step, not the declared one - describe_bars divides by
    # this, and src_step now defaults to None
    out.attrs.update(tf=tf, src_step=src, label=label,
                     n_src=n_src, dropped_partial=dropped_partial)
    return out


def describe_bars(df, label, tf):
    """Full coverage + integrity report for one set. Returns it unchanged."""
    n_src   = df.attrs.get("n_src")
    span    = (df.index[-1] - df.index[0]).total_seconds() / 86400
    bad_rng = int((~((df["high"] >= df[["open", "close"]].max(axis=1) - 1e-6) &
                     (df["low"]  <= df[["open", "close"]].min(axis=1) + 1e-6))).sum())
    print(f"{label}  {tf}")
    print(f"  bars      : {len(df):,}")
    print(f"  coverage  : {df.index[0]}  ->  {df.index[-1]}   ({span:,.1f} days)")
    print(f"  close     : {df['close'].min():,.2f} - {df['close'].max():,.2f}")
    print(f"  volume    : {df['volume'].sum():,.0f}"
          + ("   [!] all-zero: source reports no volume" if df["volume"].sum() == 0 else ""))
    print(f"  integrity : high<low {int((df['high'] < df['low']).sum())} | "
          f"OHLC out of range {bad_rng} | NaN {int(df.isna().sum().sum())} | "
          f"dup ts {int(df.index.duplicated().sum())}")
    if n_src is not None and len(n_src):
        full = int(pd.Timedelta(TIMEFRAMES[tf]) / pd.Timedelta(df.attrs["src_step"]))
        print(f"  bin fill  : median {n_src.median():.0f}/{full} source bars, "
              f"min {n_src.min():.0f}, thin(<50%) {int((n_src < full * 0.5).sum()):,}")
    if df.attrs.get("dropped_partial"):
        print(f"  [note] trailing incomplete {tf} bar dropped")
    return df


def summarise(df, label, tf):
    """One compact line per asset, for the batch cellblocks below."""
    flags = []
    if (df["high"] < df["low"]).any():               flags.append("high<low")
    if df.index.duplicated().any():                  flags.append("dup-ts")
    if df[["open", "high", "low", "close"]].isna().any().any(): flags.append("NaN")
    if df["volume"].sum() == 0:                      flags.append("no-volume")
    if df.attrs.get("dropped_partial"):              flags.append("partial-dropped")
    print(f"  {label:6}{tf:>5}  {len(df):>9,} bars   "
          f"{df.index[0]:%Y-%m-%d %H:%M} -> {df.index[-1]:%Y-%m-%d %H:%M}   "
          f"{'| ' + ', '.join(flags) if flags else 'clean'}")
    return df


def _clean(raw, label, ticker):
    """Flatten yfinance output to a sorted, deduped, tz-aware OHLCV frame."""
    if raw is None or len(raw) == 0:
        raise RuntimeError(f"{label} ({ticker}): yfinance returned no rows")
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.droplevel(-1)
    df = raw.rename(columns=str.lower)
    missing = [c for c in OHLCV if c not in df.columns]
    if missing:
        raise RuntimeError(f"{label} ({ticker}): missing columns {missing}")
    df = df[OHLCV].copy()
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df = df.dropna(subset=["open", "high", "low", "close"])
    df["volume"] = df["volume"].fillna(0)
    df.index.name = "ts"
    df.columns.name = None
    return df


def fetch_daily(ticker, label):
    """Maximum available daily history. This is the only set with real depth."""
    df = _clean(yf.download(ticker, period="max", interval="1d", auto_adjust=False,
                            progress=False, threads=False), label, ticker)
    print(f"  {label:6}  1d  {len(df):>9,} bars  "
          f"{df.index[0]:%Y-%m-%d} -> {df.index[-1]:%Y-%m-%d}  "
          f"({(df.index[-1] - df.index[0]).days / 365.25:5.1f} yrs)")

    # Yahoo's daily futures bars contain bars whose open/close sit OUTSIDE the
    # high/low range. Reported, never silently repaired - a backtest that assumes
    # low <= open,close <= high would compute impossible fills on these dates.
    bad = ~((df["high"] >= df[["open", "close"]].max(axis=1) - 1e-6) &
            (df["low"]  <= df[["open", "close"]].min(axis=1) + 1e-6))
    if bad.any():
        print(f"          [!] {int(bad.sum())} bar(s) with OHLC outside the high/low "
              f"range ({100 * bad.mean():.2f}%) - source defect, NOT repaired. "
              f"Last: {df.index[bad][-1]:%Y-%m-%d}.")
    return df


YF_1M_LOOKBACK_DAYS = 30      # Yahoo keeps roughly this much 1m history
YF_1M_CHUNK_DAYS    = 7       # and hands back at most 8 days per request


@contextmanager
def _quiet_yf():
    """Mute yfinance's own logger for the duration of a request.

    It logs "$TSLA: possibly delisted; no price data found" whenever a request
    window contains no session at all. The usual cause is mundane: the oldest
    chunk of the walk lands on a Saturday/Sunday, and an RTH-only symbol has no
    minutes there. TSLA is not delisted. Empty windows are counted and reported
    by fetch_1m itself, so nothing is hidden - it is just not shouted. A symbol
    that really has no data still fails loudly, because every window comes back
    empty and the RuntimeError below fires.
    """
    lg   = logging.getLogger("yfinance")
    prev = lg.level
    lg.setLevel(logging.CRITICAL)
    try:
        yield
    finally:
        lg.setLevel(prev)


def fetch_1m(ticker, label, lookback=YF_1M_LOOKBACK_DAYS,
             chunk=YF_1M_CHUNK_DAYS, pause=0.4):
    """Walk Yahoo's 1m window in chunks, since it caps each request at 8 days."""
    end         = pd.Timestamp.utcnow().normalize() + pd.Timedelta(days=1)
    start_floor = end - pd.Timedelta(days=lookback)
    parts, empty, fails, win_end = [], [], 0, end

    while win_end > start_floor:
        win_start = max(win_end - pd.Timedelta(days=chunk), start_floor)
        try:
            with _quiet_yf():
                raw = yf.download(ticker, start=win_start.date(), end=win_end.date(),
                                  interval="1m", auto_adjust=False, progress=False,
                                  threads=False)
            if raw is not None and len(raw):
                parts.append(_clean(raw, label, ticker))
            else:
                empty.append((win_start, win_end))
        except Exception:
            fails += 1                       # a real request error, not an empty window
        win_end = win_start
        time.sleep(pause)

    if not parts:
        raise RuntimeError(f"{label} ({ticker}): no 1m data in the last {lookback}d "
                           f"({len(empty)} empty window(s), {fails} error(s)) - "
                           "this one really does have no data")
    df = pd.concat(parts).sort_index()
    df = df[~df.index.duplicated(keep="last")]

    # yfinance's `end` is exclusive, so a window is "all weekend" when no
    # business day falls in [start, end).
    closed = sum(1 for a, b in empty
                 if len(pd.bdate_range(a.date(), b.date() - pd.Timedelta(days=1))) == 0)
    note = ""
    if empty:
        note += f"   [{len(empty)} empty window(s)"
        note += f", {closed} weekend/holiday]" if closed else "]"
    if fails:
        note += f"   [{fails} request error(s)]"
    print(f"  {label:6}  1m  {len(df):>9,} bars  "
          f"{df.index[0]:%Y-%m-%d %H:%M} -> {df.index[-1]:%Y-%m-%d %H:%M}" + note)
    return df


# ---- {asset}_1d  -  daily, maximum history. Deepest data in the system -------
# Kept for regime work and long-horizon checks. NOT usable by the cellblock.2
# entry rule: that needs a bar inside the hour before the cash open, and a daily
# bar has no inside. The chart grids exclude 1d for exactly that reason.
print("daily (period='max')")
for _var, _lab, _tkr in ASSETS:
    globals()[f"{_var}_1d"] = fetch_daily(_tkr, _lab)

# ---- {asset}_1h  -  720 days of hourly bars, every asset ---------------------
# This is the set the strategy actually trades: long enough to mean something
# (~2 yrs) and available for ALL ten, including TSLA/NVDA/AAPL/VIX, whose 1m
# window is only ~30 days. Module.2 cellblocks 2-11 build the same frames one
# at a time with a chart each; this rebuilds them in one pass so the data layer
# stands alone - if those cells were skipped, the grids would silently show
# only the assets that happened to be loaded.
print(f"\n1h (period={YF_PERIOD}, interval={YF_INTERVAL})")
for _var, _lab, _tkr in ASSETS:
    _df = fetch_1h(_tkr, _lab)
    globals()[f"{_var}_1h"] = _df
    _zero = "   [!] no volume reported" if _df["volume"].sum() == 0 else ""
    print(f"  {_lab:6}  1h  {len(_df):>9,} bars  "
          f"{_df.index[0]:%Y-%m-%d %H:%M} -> {_df.index[-1]:%Y-%m-%d %H:%M}  "
          f"({(_df.index[-1] - _df.index[0]).days:>4d} days){_zero}")

# ---- {asset}_1m  -  the bases cellblocks 5-9 aggregate up from ---------------
if "nq_1m" not in globals():
    raise RuntimeError("run Module.2 cellblock.1 first - it builds nq_1m from the local CSV")

print("\n1m bases")
print(f"  {'/NQ':6}  1m  {len(nq_1m):>9,} bars  "
      f"{nq_1m.index[0]:%Y-%m-%d %H:%M} -> {nq_1m.index[-1]:%Y-%m-%d %H:%M}"
      f"   [local CSV, {(nq_1m.index[-1] - nq_1m.index[0]).days / 365.25:.1f} yrs]")

for _var, _lab, _tkr in ASSETS:
    if _var == "nq":
        continue                                   # already built from the local CSV
    globals()[f"{_var}_1m"] = fetch_1m(_tkr, _lab)

print(f"\nbuilt: {', '.join(f'{v}_1d' for v, _, _ in ASSETS)}")
print(f"built: {', '.join(f'{v}_1h' for v, _, _ in ASSETS)}")
print(f"built: {', '.join(f'{v}_1m' for v, _, _ in ASSETS)}")


In [ ]:
# cellblock.5

# {asset}_3m  -  resampled from each asset's 1m base.
# 3m exists ONLY here - Yahoo does not serve a 3m interval at any period.
print("3m (resampled from 1m)")
for _var, _lab, _tkr in ASSETS:
    _out = resample_ohlcv(globals()[f"{_var}_1m"], "3m", label=_lab)
    globals()[f"{_var}_3m"] = summarise(_out, _lab, "3m")

print(f"\nbuilt: {', '.join(f'{v}_3m' for v, _, _ in ASSETS)}")
# describe_bars({asset}_3m, "<label>", "3m") for the full report on any one set


In [ ]:
# cellblock.6

# {asset}_5m  -  resampled from each asset's 1m base.
print("5m (resampled from 1m)")
for _var, _lab, _tkr in ASSETS:
    _out = resample_ohlcv(globals()[f"{_var}_1m"], "5m", label=_lab)
    globals()[f"{_var}_5m"] = summarise(_out, _lab, "5m")

print(f"\nbuilt: {', '.join(f'{v}_5m' for v, _, _ in ASSETS)}")
# describe_bars({asset}_5m, "<label>", "5m") for the full report on any one set


In [ ]:
# cellblock.7

# {asset}_15m  -  resampled from each asset's 1m base.
# NOTE: this supersedes Module.2's native 15m pull. For /NQ that is 366k bars
# back to 2010 instead of Yahoo's 60-day window.
print("15m (resampled from 1m)")
for _var, _lab, _tkr in ASSETS:
    _out = resample_ohlcv(globals()[f"{_var}_1m"], "15m", label=_lab)
    globals()[f"{_var}_15m"] = summarise(_out, _lab, "15m")

print(f"\nbuilt: {', '.join(f'{v}_15m' for v, _, _ in ASSETS)}")
# describe_bars({asset}_15m, "<label>", "15m") for the full report on any one set


In [ ]:
# cellblock.8

# {asset}_30m  -  resampled from each asset's 1m base.
print("30m (resampled from 1m)")
for _var, _lab, _tkr in ASSETS:
    _out = resample_ohlcv(globals()[f"{_var}_1m"], "30m", label=_lab)
    globals()[f"{_var}_30m"] = summarise(_out, _lab, "30m")

print(f"\nbuilt: {', '.join(f'{v}_30m' for v, _, _ in ASSETS)}")
# describe_bars({asset}_30m, "<label>", "30m") for the full report on any one set


In [ ]:
# cellblock.9

# {asset}_60m  -  resampled from each asset's 1m base.
print("60m (resampled from 1m)")
for _var, _lab, _tkr in ASSETS:
    _out = resample_ohlcv(globals()[f"{_var}_1m"], "60m", label=_lab)
    globals()[f"{_var}_60m"] = summarise(_out, _lab, "60m")

print(f"\nbuilt: {', '.join(f'{v}_60m' for v, _, _ in ASSETS)}")
# describe_bars({asset}_60m, "<label>", "60m") for the full report on any one set


In [ ]:
# cellblock.10

# DATASETS registry + integrity / conservation verification
TFS = ["1d", "1m", "3m", "5m", "15m", "30m", "60m"]

DATASETS = {lab: {tf: globals()[f"{var}_{tf}"]
                  for tf in TFS if f"{var}_{tf}" in globals()}
            for var, lab, _ in ASSETS}

print("bar counts")
print(f"  {'asset':6}" + "".join(f"{tf:>11}" for tf in TFS))
print("  " + "-" * (6 + 11 * len(TFS)))
for _lab, _sets in DATASETS.items():
    print(f"  {_lab:6}" + "".join(
        f"{len(_sets[tf]):>11,}" if tf in _sets else f"{'-':>11}" for tf in TFS))

# ---- integrity -------------------------------------------------------------
problems = []
for _lab, _sets in DATASETS.items():
    for _tf, _df in _sets.items():
        if len(_df) == 0:
            problems.append(f"{_lab} {_tf}: EMPTY")
        if _df.index.duplicated().any():
            problems.append(f"{_lab} {_tf}: duplicate timestamps")
        if not _df.index.is_monotonic_increasing:
            problems.append(f"{_lab} {_tf}: not sorted")
        if (_df["high"] < _df["low"]).any():
            problems.append(f"{_lab} {_tf}: high < low")
        if _df[["open", "high", "low", "close"]].isna().any().any():
            problems.append(f"{_lab} {_tf}: NaN in OHLC")
        if list(_df.columns)[:5] != OHLCV:
            problems.append(f"{_lab} {_tf}: unexpected columns {list(_df.columns)}")
        # every resampled bar must start on the timeframe grid
        if _tf in TIMEFRAMES and _tf != "1m" and len(_df) > 2:
            _step = pd.Timedelta(TIMEFRAMES[_tf])
            if ((_df.index.to_series().diff().dropna() % _step)
                    != pd.Timedelta(0)).any():
                problems.append(f"{_lab} {_tf}: bars not aligned to the {_tf} grid")

print("\nintegrity:", "NONE" if not problems else "")
for p in problems:
    print(f"  [!] {p}")

# ---- volume conservation: aggregation must not create or destroy contracts --
print("\nvolume conservation (1m -> each resampled set):")
for _lab, _sets in DATASETS.items():
    if "1m" not in _sets:
        continue
    _base = _sets["1m"]["volume"].sum()
    _bad = [tf for tf in RESAMPLED
            if tf in _sets and not _sets[tf].attrs.get("dropped_partial")
            and _sets[tf]["volume"].sum() != _base]
    print(f"  {_lab:6} {'OK' if not _bad else 'FAIL -> ' + str(_bad)}"
          f"   (sets with a dropped partial bar are exempt)")

# ---- no-lookahead: a bar may only contain 1m bars from inside its own window -
print("\nno-lookahead spot check (25 random 15m bars per asset):")
_rng = np.random.default_rng(0)
for _lab, _sets in DATASETS.items():
    if "15m" not in _sets or "1m" not in _sets:
        continue
    _b, _r = _sets["1m"], _sets["15m"]
    _ok = True
    for _p in _rng.choice(len(_r), size=min(25, len(_r)), replace=False):
        _t = _r.index[_p]
        _w = _b.loc[_t : _t + pd.Timedelta("15min") - pd.Timedelta("1s")]
        if len(_w) == 0:
            continue
        if not (abs(_r["open"].iloc[_p]  - _w["open"].iloc[0])  < 1e-3 and
                abs(_r["close"].iloc[_p] - _w["close"].iloc[-1]) < 1e-3 and
                abs(_r["high"].iloc[_p]  - _w["high"].max())     < 1e-3 and
                abs(_r["low"].iloc[_p]   - _w["low"].min())      < 1e-3 and
                _r["volume"].iloc[_p] == _w["volume"].sum()):
            _ok = False
            break
    print(f"  {_lab:6} {'OK - every bar built only from 1m bars inside its window'
                        if _ok else 'FAIL - bar contains data from outside its window'}")


## Module.4
### Strategy importing

In [ ]:
# cellblock.1

# run the strategy  -  the whole pipeline, end to end, on real data.
#
# This cellblock used to hold a SECOND copy of coin_flip_signals() and its
# helpers. That copy re-defined all six names from Module.3 cellblock.1, so
# whichever cell ran last silently won. The strategy now lives in Module.3
# only; this cellblock calls it instead of redefining it.
#
# It also SCREENS the datasets before handing them to a grid, because two kinds
# of set will otherwise end the run:
#   tz-naive  - yfinance returns naive stamps for DAILY bars and tz-aware ones
#               for intraday, so every {asset}_1d lands here without a zone.
#               run_session_trades places entries by wall clock, so a naive
#               index raises "Cannot convert tz-naive timestamps" and, in an
#               unguarded grid, takes every remaining panel down with it. That
#               is why a run can show two panels and then stop.
#   daily     - a 1d bar cannot contain the hour before the cash open, so even
#               localized it can never produce a trade.
# Both are dropped BY NAME below, never silently.
#
# Read plot_fan_grid FIRST. A single equity curve shows what one coin happened
# to do; the strategy is a random variable, and the fan is the only chart that
# says whether any of it means anything.
_need = ["collect_assets", "plot_pnl_grid", "plot_fan_grid", "coin_flip_signals",
         "run_session_trades", "INSTRUMENTS"]
_missing = [n for n in _need if n not in globals()]
if _missing:
    raise RuntimeError(f"run Module.3 cellblocks 1-3 first - missing {_missing}")

# which Module.2 cellblock builds each asset's 1h set, for the coverage report
_M2_CELL = {sym: n for n, sym in enumerate(
    ["/NQ", "/ES", "/RTY", "/YM", "BTC", "ETH", "TSLA", "NVDA", "AAPL", "VIX"], start=2)}

_raw = collect_assets(quiet=True)

sets, _dropped = {}, []
for _lab, (_sym, _bars) in _raw.items():
    if _bars.index.tz is None:
        _dropped.append(f"{_lab} [tz-naive]")
    elif _lab.endswith(" 1d"):
        _dropped.append(f"{_lab} [daily - no bar inside the entry window]")
    else:
        sets[_lab] = (_sym, _bars)

print(f"plotting {len(sets)} datasets across "
      f"{len({s for s, _ in sets.values()})} of {len(INSTRUMENTS)} assets")
for _lab, (_sym, _b) in sets.items():
    print(f"  {_lab:12} {len(_b):>10,} bars  {_b.index[0]:%Y-%m-%d} -> "
          f"{_b.index[-1]:%Y-%m-%d}  ({(_b.index[-1]-_b.index[0]).days/365.25:5.1f} yrs)"
          + ("" if INSTRUMENTS[_sym]["tradable"] else "   [not tradable]"))
if _dropped:
    print(f"  dropped: {', '.join(_dropped)}")

# Coverage: an asset absent from the grid is almost always an unrun Module.2
# cellblock, not a strategy result. Say so rather than quietly drawing fewer.
_have = {s for s, _ in sets.values()}
_gone = [s for s in INSTRUMENTS if s not in _have]
if _gone:
    print("\n  [!] no dataset for " + ", ".join(_gone))
    print("      run Module.2 " + ", ".join(
        f"cellblock.{_M2_CELL[s]} ({s})" for s in _gone if s in _M2_CELL)
        + "  - each one builds that asset's 1h set")
if not sets:
    raise RuntimeError("nothing left to plot - run Module.2 cellblocks 1-11 first")
print()

one_coin = plot_pnl_grid(sets)   # seed FLIP_SEED only - the tempting picture
print(one_coin.round(0).to_string())
print()

fifty = plot_fan_grid(sets)      # COIN_SEEDS coins - the honest one
print()
print(fifty.round(0).to_string())

# Per-dataset detail, when one panel is worth a closer look:
#   tr, bk, _ = _run_one(nq_1m, "/NQ")
#   plot_pnl(tr, bk, "/NQ")
